In [0]:
%sql

create or replace table workspace.silver.spot_prices
comment '''
    one row per price area and utc hour. price_per_kwh is in the main currency unit (nok/kwh, sek/kwh). notes: 
    (1) units differ by source: nok/mwh and ore/kwh, normalised to per kwh. 
    (2) unit is missing for all of no5 and sporadically elsewhere: inferred from currency, is_unit_inferred = 1. assumes no5 is nok/mwh, based on scale only. 
    (3) currencies differ by country: kept, not converted. the load fails on an unknown currency or a unit that contradicts its currency.
'''
as

select
    p.price_area,
    cast(p.hour_utc as timestamp) as hour_utc,
    cast(cast(p.price as double) / case
        when lower(p.currency) = 'nok' and lower(coalesce(p.unit, 'NOK/MWh')) = 'nok/mwh' then 1000
        when lower(p.currency) = 'sek' and lower(coalesce(p.unit, 'ore/kWh')) = 'ore/kwh' then 100
        else raise_error(concat_ws(' ', 'unexpected currency/unit:', p.currency, p.unit))
    end as decimal(12, 5)) as price_per_kwh,
    p.currency,
    if(p.unit is null, 1, 0) as is_unit_inferred,
    current_timestamp() as _processed_at
from 
    workspace.bronze.spot_prices as p